# 🎬 动漫数据分析 — 交互式分析 Notebook

## 研究问题
1. 动漫评分的分布规律是什么？哪些因素影响评分？
2. 能否基于特征预测动漫评分？
3. 能否将动漫自动分群？

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

from src.cleaner import clean_all
from src.features import prepare_modeling_features, get_top_genres
from src.model import train_regression_models, get_feature_importance, run_clustering, run_pca
from src.visualize import plot_distribution, plot_scatter, plot_correlation_heatmap

print('✅ 所有模块加载成功')

## 1. 数据加载与清洗

In [ ]:
# 加载原始数据
df_raw = pd.read_excel('../data/AAACG.xlsx')
print(f'原始数据: {df_raw.shape[0]} 行 × {df_raw.shape[1]} 列')
print(f'\n列名: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# 缺失值分析
null_info = pd.DataFrame({
    '缺失数': df_raw.isnull().sum(),
    '缺失率%': (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
}).sort_values('缺失率%', ascending=False)
print(null_info[null_info['缺失数'] > 0])

In [ ]:
# 执行清洗
df = clean_all('../data/AAACG.xlsx')
df.head()

## 2. 探索性数据分析 (EDA)

In [ ]:
# 评分分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['分数'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['分数'].mean(), color='red', linestyle='--', label=f'均值: {df["分数"].mean():.2f}')
axes[0].set_xlabel('评分')
axes[0].set_ylabel('频次')
axes[0].set_title('动漫评分分布')
axes[0].legend()

df['分数'].plot.box(ax=axes[1], patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1].set_title('评分箱线图')

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 类型
all_genres = []
for genres in df['类型_列表']:
    all_genres.extend([g.strip() for g in genres if g.strip()])

genre_counts = pd.Series(all_genres).value_counts().head(15)
fig, ax = plt.subplots(figsize=(12, 6))
genre_counts.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('数量')
ax.set_title('Top 15 动漫类型')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 年份趋势
df_year = df[df['年份'] >= 1990]
yearly = df_year.groupby('年份').agg(
    平均评分=('分数', 'mean'),
    产出数量=('番ID', 'count')
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(yearly.index, yearly['产出数量'], color='steelblue', alpha=0.6, label='产出数量')
ax1.set_xlabel('年份')
ax1.set_ylabel('产出数量', color='steelblue')

ax2 = ax1.twinx()
ax2.plot(yearly.index, yearly['平均评分'], color='red', marker='o', markersize=3, label='平均评分')
ax2.set_ylabel('平均评分', color='red')

ax1.set_title('动漫产出数量与平均评分随年份变化')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

In [ ]:
# 相关性热力图
numeric_cols = ['分数', '集数', '时长_分钟', '人气排名', '收藏', '关注人数', '社区成员数']
existing = [c for c in numeric_cols if c in df.columns]
corr = df[existing].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('数值特征相关性热力图')
plt.tight_layout()
plt.show()

## 3. 特征工程

In [ ]:
df_model, feature_cols = prepare_modeling_features(df)
print(f'特征数量: {len(feature_cols)}')
print(f'特征列: {feature_cols}')

## 4. 建模 — 回归预测评分

In [ ]:
results = train_regression_models(df_model, feature_cols, target='分数')

In [ ]:
# 特征重要性
df_imp = get_feature_importance(results, feature_cols)
top_imp = df_imp['平均重要性'].head(15)

fig, ax = plt.subplots(figsize=(10, 8))
top_imp.plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('重要性')
ax.set_title('Top 15 特征重要性（预测评分）')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. 建模 — 聚类分析

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df_model[feature_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_clustered, km = run_clustering(df_model, feature_cols, n_clusters=4)

In [ ]:
# PCA 可视化
X_pca, pca = run_pca(X_scaled, n_components=2)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=df_clustered['聚类标签'], cmap='Set1',
                     alpha=0.4, s=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
ax.set_title('动漫聚类 PCA 可视化')
plt.colorbar(scatter, ax=ax, label='聚类标签')
plt.tight_layout()
plt.show()

In [ ]:
# 各聚类特征对比
key_cols = ['分数', '集数', '收藏', '关注人数', '社区成员数']
key_cols = [c for c in key_cols if c in df_clustered.columns]
cluster_means = df_clustered.groupby('聚类标签')[key_cols].mean()
print('各聚类特征均值:')
print(cluster_means.to_string())

# 归一化对比图
cluster_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())
fig, ax = plt.subplots(figsize=(10, 6))
cluster_norm.T.plot.bar(ax=ax, width=0.8)
ax.set_ylabel('归一化值')
ax.set_title('各聚类特征对比（归一化）')
ax.legend(title='聚类', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 6. 大模型辅助分析

使用 LLM API 对分析结果生成结构化总结。

In [ ]:
import json
from src.llm_analysis import run_llm_analysis

llm_result = run_llm_analysis(df_clustered, results)
print(json.dumps(llm_result, ensure_ascii=False, indent=2))

## 7. 结论

### 关键发现
1. **收藏数是预测评分的最强特征**，贡献了约 69% 的模型重要性
2. **喜剧、动作、奇幻**是最常见的动漫类型
3. **GradientBoosting** 模型表现最佳，R²=0.73
4. 聚类分析识别出 4 类动漫群体，特征差异明显

### 局限性
- 数据缺失较多（受众群体 70%、年份 68%）
- 模型 R²=0.73 仍有提升空间